In [1]:
# notebooks/03_Test.ipynb

# 1. Environment Initialization
import os
import shutil
import sys
import random
import itertools
import numpy as np
import pandas as pd
import torch
from torch.amp import autocast
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q monai
from monai.inferers import sliding_window_inference

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.8 MB/s eta 0:00:00


In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

PROJECT_ROOT = "/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor"
sys.path.append(PROJECT_ROOT)

import src.config as config
from src.dataset import get_test_dataloader
from src.metrics import SegmentationMetrics
from src.models.mamba_backbone import MambaBackbone, SharedDeepMambaBackbone
from src.models.fusion import PresenceAwareCrossModalFusion
from src.models.decoder import SegmentationDecoder3D, AuxiliaryDecoder3D

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
DATASET_ZIP = "/content/drive/MyDrive/ML-Datasets/BraTS2020_TrainingData_ABC.zip"
LOCAL_EXTRACT_DIR = "/content/MICCAI_BraTS2020_TrainingData_ABC"
LOCAL_DATA_DIR = LOCAL_EXTRACT_DIR

# Verify archive existence immediately before runtime allocation
assert os.path.exists(DATASET_ZIP), f"Dataset archive not found: {DATASET_ZIP}"

# Robust extraction guard: triggers if directory does not exist or is completely empty
if not os.path.exists(LOCAL_DATA_DIR) or len(os.listdir(LOCAL_DATA_DIR)) == 0:
    print(f"Extracting preprocessed dataset to local runtime storage: {LOCAL_EXTRACT_DIR}...")
    os.makedirs(LOCAL_EXTRACT_DIR, exist_ok=True)
    shutil.unpack_archive(DATASET_ZIP, LOCAL_EXTRACT_DIR, "zip")
    print("Extraction complete.")
else:
    print("Existing local preprocessed dataset detected. Skipping extraction.")

Extracting preprocessed dataset to local runtime storage: /content/MICCAI_BraTS2020_TrainingData_ABC...
Extraction complete.


In [5]:
import importlib

importlib.reload(config)

import src.engine as engine
importlib.reload(engine)

import src.models.mamba_backbone as mbb
importlib.reload(mbb)

import src.models.fusion as mf
importlib.reload(mf)

import src.models.decoder as md
importlib.reload(md)

import src.losses as sl
importlib.reload(sl)

import src.transforms as tf
importlib.reload(tf)

print("Reloaded all src files.")

Reloaded all src files.


In [6]:
# 2. Pipeline Dataset Loaders Construction
test_loader = get_test_dataloader()

# 3. Structural Module Instantiations & Checkpoint Loading
backbone = MambaBackbone(embed_dim=config.EMBED_DIM).to(device)
fusion = PresenceAwareCrossModalFusion(embed_dim=config.EMBED_DIM).to(device)
shared_backbone = SharedDeepMambaBackbone(embed_dim=config.EMBED_DIM).to(device)
decoder = SegmentationDecoder3D(embed_dim=config.EMBED_DIM, out_channels=config.NUM_SEG_CLASSES).to(device)
aux_decoder = AuxiliaryDecoder3D(embed_dim=config.EMBED_DIM, out_channels=config.NUM_SEG_CLASSES).to(device)

best_seg_path = os.path.join(config.CHECKPOINT_DIR, "best_seg.pth")
assert os.path.exists(best_seg_path), f"Checkpoint not found: {best_seg_path}"

checkpoint = torch.load(best_seg_path, map_location=device)
backbone.load_state_dict(checkpoint["backbone_state"])
fusion.load_state_dict(checkpoint["fusion_state"])
shared_backbone.load_state_dict(checkpoint["shared_backbone_state"])
decoder.load_state_dict(checkpoint["decoder_state"])
aux_decoder.load_state_dict(checkpoint["aux_decoder_state"])

backbone.eval()
fusion.eval()
shared_backbone.eval()
decoder.eval()

print("Checkpoint loaded.")

Checkpoint loaded.


In [7]:
# 4. Define 15 Modality Combinations
# 0: T1, 1: T1ce, 2: T2, 3: FLAIR
mod_idx = [0, 1, 2, 3]
mod_names = ["T1", "T1ce", "T2", "FLAIR"]

combinations = []
for r in range(1, 5):
    combinations.extend(list(itertools.combinations(mod_idx, r)))

# --- Resumption and Directory Setup ---
csv_out_path = os.path.join(config.RESULT_DIR, "modality_dropout_results.csv")

completed_combinations = []
if os.path.exists(csv_out_path):
    existing_df = pd.read_csv(csv_out_path)
    if "Present Modalities" in existing_df.columns:
        completed_combinations = existing_df["Present Modalities"].tolist()
        print(f"[*] Found existing results. Resuming... ({len(completed_combinations)}/15 completed)")
# --------------------------------------

results = []

In [8]:
# 5. Evaluation Loop across 15 settings
print(f"Starting multi-modality evaluation across {len(combinations)} settings...")

with torch.no_grad():
    for comb in combinations:
        comb_names = "+".join([mod_names[i] for i in comb])

        # --- Skip if already processed ---
        if comb_names in completed_combinations:
            print(f"Skipping {comb_names} (already completed)...")
            continue
        # ---------------------------------

        print(f"\n--- Testing Modalities: {comb_names} ---")

        keep_mask = torch.zeros(4, device=device)
        keep_mask[list(comb)] = 1.0

        seg_tracker = SegmentationMetrics()

        for batch in tqdm(test_loader, desc=f"Eval {comb_names}", leave=False):
            images = batch["image"].to(device)
            seg_targets = batch["label"].to(device)
            B_current = images.size(0)
            batch_seg_logits = []

            for b in range(B_current):
                single_img = images[b:b+1]

                def evaluation_predictor(patch_images):
                    # 1. Apply dropout to the raw input images FIRST
                    processed_images = patch_images.clone()
                    for i in range(4):
                        if keep_mask[i] == 0.0:
                            processed_images[:, i, :, :, :] = 0.0

                    # 2. Backbone now processes genuinely incomplete inputs
                    modality_tokens, spatial_shape, skip_features, _ = backbone(processed_images)

                    # 3. Format tokens for the fusion module as required
                    processed_modality_tokens = []
                    for i in range(4):
                        if keep_mask[i] == 1.0:
                            processed_modality_tokens.append(modality_tokens[i])
                        else:
                            processed_modality_tokens.append(torch.zeros_like(modality_tokens[i]))

                    fused_tokens = fusion(processed_modality_tokens, processed_images)
                    latent_tokens = shared_backbone(fused_tokens)
                    seg_logits = decoder(latent_tokens, spatial_shape, skip_features)

                    return seg_logits

                with autocast(device_type=device.type, enabled=(device.type == "cuda")):
                    seg_logits = sliding_window_inference(
                        inputs=single_img,
                        roi_size=config.PATCH_SIZE,
                        sw_batch_size=24,
                        predictor=evaluation_predictor,
                        overlap=0.5,
                        mode="gaussian"
                    )
                batch_seg_logits.append(seg_logits)

            seg_logits = torch.cat(batch_seg_logits, dim=0)
            seg_preds = torch.argmax(seg_logits, dim=1, keepdim=True)
            seg_tracker.update(seg_preds, seg_targets, run_hd=False)

        metrics = seg_tracker.compute(run_hd=False)
        mean_dice = (metrics["dice_WT"] + metrics["dice_TC"] + metrics["dice_ET"]) / 3.0

        current_result = {
            "Missing Modalities": "+".join([mod_names[i] for i in mod_idx if i not in comb]) or "None",
            "Present Modalities": comb_names,
            "Dice WT": round(metrics["dice_WT"], 4),
            "Dice TC": round(metrics["dice_TC"], 4),
            "Dice ET": round(metrics["dice_ET"], 4),
            "Mean Dice": round(mean_dice, 4),
            "HD95 WT": round(metrics["hd95_WT"], 4) if "hd95_WT" in metrics else "N/A",
            "HD95 TC": round(metrics["hd95_TC"], 4) if "hd95_TC" in metrics else "N/A",
            "HD95 ET": round(metrics["hd95_ET"], 4) if "hd95_ET" in metrics else "N/A"
        }

        results.append(current_result)
        print(f"Mean Dice: {mean_dice:.4f} | WT: {metrics['dice_WT']:.4f}, TC: {metrics['dice_TC']:.4f}, ET: {metrics['dice_ET']:.4f}")

        # --- Incremental Save to CSV ---
        current_df = pd.DataFrame([current_result])
        if not os.path.exists(csv_out_path):
            current_df.to_csv(csv_out_path, index=False)
        else:
            current_df.to_csv(csv_out_path, mode='a', header=False, index=False)
        # -------------------------------

Starting multi-modality evaluation across 15 settings...

--- Testing Modalities: T1 ---


Mean Dice: 0.2223 | WT: 0.2973, TC: 0.3694, ET: 0.0002

--- Testing Modalities: T1ce ---


Mean Dice: 0.5600 | WT: 0.3690, TC: 0.6261, ET: 0.6849

--- Testing Modalities: T2 ---


Mean Dice: 0.4966 | WT: 0.6009, TC: 0.5712, ET: 0.3176

--- Testing Modalities: FLAIR ---


Mean Dice: 0.3750 | WT: 0.7705, TC: 0.3117, ET: 0.0429

--- Testing Modalities: T1+T1ce ---


...

Mean Dice: 0.7390 | WT: 0.5913, TC: 0.8114, ET: 0.8144

--- Testing Modalities: T1+T1ce+FLAIR ---


Mean Dice: 0.8196 | WT: 0.8560, TC: 0.8048, ET: 0.7979

--- Testing Modalities: T1+T2+FLAIR ---


Mean Dice: 0.5218 | WT: 0.8458, TC: 0.5524, ET: 0.1673

--- Testing Modalities: T1ce+T2+FLAIR ---


Mean Dice: 0.8085 | WT: 0.8797, TC: 0.7968, ET: 0.7490

--- Testing Modalities: T1+T1ce+T2+FLAIR ---


Mean Dice: 0.8462 | WT: 0.8888, TC: 0.8276, ET: 0.8220


In [9]:
# 6. Display Results
print("\nTesting finished. Loading final combined results:")
if os.path.exists(csv_out_path):
    df_results = pd.read_csv(csv_out_path)
    display(df_results)
else:
    print("No results found.")


Testing finished. Loading final combined results:


,Missing Modalities,Present Modalities,Dice WT,Dice TC,Dice ET,Mean Dice,HD95 WT,HD95 TC,HD95 ET
0,T1ce+T2+FLAIR,T1,0.2973,0.3694,0.0002,0.2223,0.0,0.0,0.0
1,T1+T2+FLAIR,T1ce,0.3690,0.6261,0.6849,0.5600,0.0,0.0,0.0
2,T1+T1ce+FLAIR,T2,0.6009,0.5712,0.3176,0.4966,0.0,0.0,0.0
3,T1+T1ce+T2,FLAIR,0.7705,0.3117,0.0429,0.3750,0.0,0.0,0.0
4,T2+FLAIR,T1+T1ce,0.4620,0.7468,0.7491,0.6526,0.0,0.0,0.0
5,T1ce+FLAIR,T1+T2,0.5430,0.5015,0.1841,0.4095,0.0,0.0,0.0
6,T1ce+T2,T1+FLAIR,0.8274,0.4711,0.0086,0.4357,0.0,0.0,0.0
7,T1+FLAIR,T1ce+T2,0.5550,0.7738,0.7447,0.6912,0.0,0.0,0.0
8,T1+T2,T1ce+FLAIR,0.8147,0.7409,0.7662,0.7739,0.0,0.0,0.0
9,T1+T1ce,T2+FLAIR,0.8588,0.5989,0.3463,0.6013,0.0,0.0,0.0
